# Training of ML solution

#### [Applying machine learning on sensor data for irrigation recommendations: revealing the agronomist’s tacit knowledge](https://link.springer.com/article/10.1007/s11119-017-9527-4), [Precision Agriculture](https://link.springer.com/journal/11119), 2018

In [33]:
import os
from datetime import datetime
from math import log
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.patches as mpatches
from matplotlib.ticker import MaxNLocator
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.ensemble import GradientBoostingRegressor, GradientBoostingClassifier
from sklearn.metrics import mean_squared_error, r2_score, accuracy_score, f1_score, classification_report, make_scorer


In [34]:
weather_file = os.path.join("/", "home", "data", "ml_data", "raw_data", "weather_1day.csv")
irrigation_groundtruth_file = os.path.join("/", "home", "data", "ml_data", "raw_data", "irrigation_1day.csv")
soil_moisture_file = os.path.join("/", "home", "data", "ml_data", "raw_data", "soil_moisture_1day.csv")

weather_dataset = pd.read_csv(weather_file)
irrigation_dataset = pd.read_csv(irrigation_groundtruth_file)
soil_moisture_dataset = pd.read_csv(soil_moisture_file)

#### Preprocessing data

In [35]:
# Removing useless meteo variables
weather_dataset = weather_dataset.loc[weather_dataset["detectedValueTypeId"].isin(["AIR_HUM","AIR_TEMP","SOLAR_RAD",])]
weather_dataset["value"] = weather_dataset["value"].apply(lambda row: round(row,2))
# Pivoting weather data
weather_dataset = weather_dataset.pivot_table(
    index=["date", "fieldName", "sectorName"],
    columns="detectedValueTypeId",
    values="value"
).reset_index()
weather_dataset.dropna(subset=["AIR_HUM"], inplace=True)


soil_moisture_dataset["yy"] = soil_moisture_dataset["yy"].apply(lambda x : abs(x))
soil_moisture_dataset["value"] = soil_moisture_dataset["value"].apply(lambda row: round(row,2))
# Pivoting moisture data
soil_moisture_dataset = soil_moisture_dataset.pivot_table(
    index=["date", "fieldName", "sectorName"],
    columns="yy",
    values="value"
).reset_index()

soil_moisture_dataset = soil_moisture_dataset.rename(columns={20 : 'moistureDepth20cm', 60 : 'moistureDepth60cm'})
soil_moisture_dataset.dropna(subset=["moistureDepth60cm"], inplace=True)
soil_moisture_dataset.dropna(subset=["moistureDepth20cm"], inplace=True)

# Unifying field name across years
weather_dataset["fieldName"] = "Fondo Errano"
irrigation_dataset["fieldName"] = "Fondo Errano"
soil_moisture_dataset["fieldName"] = "Fondo Errano"

# Create dataset
dataset = soil_moisture_dataset.merge(weather_dataset, on = ["date","fieldName","sectorName"], how = 'inner').merge(irrigation_dataset,on = ["date","fieldName","sectorName"], how = 'inner')
# Removing useless categorical attributes
dataset = dataset.drop(columns=["fieldName","sectorName"])

dataset["date"] = pd.to_datetime(dataset["date"])
data = dataset.sort_values("date").reset_index(drop=True)

print(f"Dataset has now {len(dataset)} rows with \n{dataset.isna().sum()}\n null values")

Dataset has now 1272 rows with 
date                 0
moistureDepth20cm    0
moistureDepth60cm    0
AIR_HUM              0
AIR_TEMP             0
SOLAR_RAD            0
irrigation           0
dtype: int64
 null values


#### Add forecasts, IPPD, saturation and drought data

In [36]:
meteo_cols = ["AIR_HUM", "AIR_TEMP", "SOLAR_RAD"]
window = 1 # days
for col in meteo_cols:
    dataset[f"{col}_forecast"] = dataset[col].shift(-1)
    dataset.loc[dataset.index[-1], f"{col}_forecast"] = dataset.loc[dataset.index[-1], col]

    dataset[f"{col}_IPPD"] = dataset[col].shift(1)
    dataset.loc[dataset.index[0], f"{col}_IPPD"] = dataset.loc[dataset.index[0], col]

for depth in [20, 60]:
    col = f"moistureDepth{depth}cm"

    dataset[f"drought_duration_{depth}cm"] = (
        (dataset[col] <= -300)
        .astype(int)
        .rolling(window=window, min_periods=1)
        .sum()
    )

    # Saturation duration negli ultimi 7 giorni
    dataset[f"saturation_duration_{depth}cm"] = (
        (dataset[col] >= -50)
        .astype(int)
        .rolling(window=window, min_periods=1)
        .sum()
    )

## Utility functions: change dataset time granularity

In [41]:
def change_time_granularity(df: pd.DataFrame, days: int) -> pd.DataFrame:
    data = df.copy()

    # Crea un indice di gruppo ogni 'days' giorni (basato sui dati esistenti, non sul calendario)
    start_date = data["date"].min()
    data["group"] = ((data["date"] - start_date).dt.days // days)

    # Aggregazioni
    agg_dict = {
        "moistureDepth20cm": "mean",
        "moistureDepth60cm": "mean",
        "AIR_HUM": "mean",
        "AIR_TEMP": "mean",
        "SOLAR_RAD": "mean",
        "irrigation": "sum",
        "AIR_HUM_forecast": "mean",
        "AIR_HUM_IPPD": "mean",
        "AIR_TEMP_forecast": "mean",
        "AIR_TEMP_IPPD": "mean",
        "SOLAR_RAD_forecast": "mean",
        "SOLAR_RAD_IPPD": "mean",
        "drought_duration_20cm": "sum",
        "saturation_duration_20cm": "sum",
        "drought_duration_60cm": "sum",
        "saturation_duration_60cm": "sum"
    }

    # Raggruppa per 'group' e aggrega
    df_grouped = data.groupby("group").agg(agg_dict)

    # Recupera la data di inizio per ogni gruppo
    df_grouped["date"] = data.groupby("group")["date"].min()

    # Rimetti in ordine le colonne
    df_grouped = df_grouped.reset_index(drop=True).sort_values("date")

    return df_grouped

def split_dataset_by_year(dataset: pd.DataFrame, year: int, classification: bool = False):
    train_mask = dataset["date"].dt.year < year
    test_mask = dataset["date"].dt.year >= year

    X_train = dataset.loc[train_mask].drop(columns=["irrigation", "date"])
    y_train = dataset.loc[train_mask, "irrigation"]

    X_test = dataset.loc[test_mask].drop(columns=["irrigation", "date"])
    y_test = dataset.loc[test_mask, "irrigation"]
    return X_train, X_test, y_train, y_test
        


### Gradient Boost Regression Tree

#### Hyperparameter optimization through AutoML - Gradient Boost Regression Trees

In [ ]:
iterations = [1, 2 ,3 ,7]

for iteration in iterations:
    df = dataset.copy(deep=True)
    if iteration > 1:
        df = change_time_granularity(df, iteration)

    X_train, X_test, y_train, y_test = split_dataset_by_year(df, 2024)

    print(f"X_train: {X_train.shape}")
    print(f"X_test: {X_test.shape}")
    print(f"y_train: {y_train.shape}")
    print(f"y_test: {y_test.shape}")

    gbr = GradientBoostingRegressor(random_state=42)

    param_dist = {
        "n_estimators": np.arange(100, 1001, 100),    # numero di alberi
        "learning_rate": np.linspace(0.01, 0.2, 20),  # tasso di apprendimento
        "max_depth": np.arange(2, 8),                 # profondità massima degli alberi
        "subsample": np.linspace(0.6, 1.0, 5),        # frazione di campioni per albero
        "min_samples_split": np.arange(2, 11),        # minimo campioni per split
        "min_samples_leaf": np.arange(1, 11)          # minimo campioni in foglia
    }

    rmse_scorer = make_scorer(lambda y_true, y_pred: -np.sqrt(mean_squared_error(y_true, y_pred)))

    random_search = RandomizedSearchCV(
        estimator=gbr,
        param_distributions=param_dist,
        n_iter=600,              # numero di combinazioni da provare
        scoring=rmse_scorer,
        cv=3,                   # cross-validation a 3 fold
        verbose=0,
        random_state=42,
        n_jobs=-1               # usa tutti i core disponibili
    )

    random_search.fit(X_train, y_train)

    print("Best parameters:", random_search.best_params_)

    best_gbr = random_search.best_estimator_

    y_pred = best_gbr.predict(X_test)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)
    print(f"Sampling for {iteration} days")
    print(f"RMSE: {rmse:.3f}")
    print(f"R²: {r2:.3f}")

X_train: (1108, 15)
X_test: (164, 15)
y_train: (1108,)
y_test: (164,)


KeyboardInterrupt: 

### Gradient Boosting Classification Tree

In [40]:
print(y_test)

1108    11.36
1109     0.00
1110     0.00
1111     0.00
1112     0.00
        ...  
1267     0.00
1268     0.00
1269     0.00
1270     0.00
1271     0.00
Name: irrigation, Length: 164, dtype: float64


In [ ]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import make_scorer, accuracy_score, f1_score, classification_report
import numpy as np

iterations = [1, 2, 3, 7]

for iteration in iterations:
    df = dataset.copy(deep=True)
    if iteration > 1:
        df = change_time_granularity(df, iteration)

    X_train, X_test, y_train, y_test = split_dataset_by_year(df, 2025, True)

    print(f"X_train: {X_train.shape}")
    print(f"X_test: {X_test.shape}")
    print(f"y_train: {y_train.shape}")
    print(f"y_test: {y_test.shape}")

    gbc = GradientBoostingClassifier(random_state=42)

    param_dist = {
        "n_estimators": np.arange(100, 801, 100),
        "learning_rate": np.linspace(0.01, 0.2, 20),
        "max_depth": np.arange(2, 8),
        "subsample": np.linspace(0.6, 1.0, 5),
        "min_samples_split": np.arange(2, 11),
        "min_samples_leaf": np.arange(1, 11)
    }

    acc_scorer = make_scorer(accuracy_score)

    random_search = RandomizedSearchCV(
        estimator=gbc,
        param_distributions=param_dist,
        n_iter=300,
        scoring=acc_scorer,
        cv=3,
        verbose=1,
        random_state=42,
        n_jobs=-1
    )

    random_search.fit(X_train, y_train)

    print("Best parameters:", random_search.best_params_)

    best_gbc = random_search.best_estimator_

    y_pred = best_gbc.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average="weighted")

    print(f"\n Sampling for {iteration} days")
    print(f"Accuracy: {acc:.3f}")
    print(f"F1-score: {f1:.3f}")
    print("\nClassification report:")
    print(classification_report(y_test, y_pred))

X_train: (1108, 15)
X_test: (164, 15)
y_train: (1108,)
y_test: (164,)
Fitting 3 folds for each of 300 candidates, totalling 900 fits


ValueError: 
All the 900 fits failed.
It is very likely that your model is misconfigured.
You can try to debug the error by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
900 fits failed with the following error:
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/site-packages/sklearn/model_selection/_validation.py", line 888, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "/usr/local/lib/python3.12/site-packages/sklearn/base.py", line 1473, in wrapper
    return fit_method(estimator, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/site-packages/sklearn/ensemble/_gb.py", line 665, in fit
    y = self._encode_y(y=y, sample_weight=None)
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/site-packages/sklearn/ensemble/_gb.py", line 1501, in _encode_y
    c